In [16]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from pathlib import Path
from langchain_core.documents import Document
from pathlib import Path
BASE_DIR = Path().resolve()  # gives you the directory where the notebook is located
from dotenv import load_dotenv
load_dotenv()


True

In [17]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [18]:
# convert to document objects for langchain
documents = [Document(page_content = chunk, metadata = {"source" : f"chunk_{i}"}) for i,chunk in enumerate(chunks)]

print("sample data : ")
for i, doc in enumerate(documents) :
    print(f"{i}. {doc}")

print("\n" + "="*50)

sample data : 
0. page_content='Microsoft acquired GitHub for 7.5 billion dollars in 2018.' metadata={'source': 'chunk_0'}
1. page_content='Tesla Cybertruck production ramp begins in 2024.' metadata={'source': 'chunk_1'}
2. page_content='Google is a large technology company with global operations.' metadata={'source': 'chunk_2'}
3. page_content='Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.' metadata={'source': 'chunk_3'}
4. page_content='SpaceX develops Starship rockets for Mars missions.' metadata={'source': 'chunk_4'}
5. page_content='The tech giant acquired the code repository platform for software development.' metadata={'source': 'chunk_5'}
6. page_content='NVIDIA designs Starship architecture for their new GPUs.' metadata={'source': 'chunk_6'}
7. page_content='Tesla Tesla Tesla financial quarterly results improved significantly.' metadata={'source': 'chunk_7'}
8. page_content='Cybertruck reser

##### SETUP : Create the three types of retrievers

### 1. Vector revtriever (Semantic Search / Dense Retrieval)

In [19]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"batch_size": 32}
)

db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"} # Note: Ensure your DB initialization doesn't have a space ("hnsw:space")
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12452.10it/s]


In [20]:
vector_retriever = db.as_retriever(search_kwargs={"k":2})

test_query = "space exploration company"
print(f"Testing :{test_query} \n")

test_docs  = vector_retriever.invoke(test_query)
for doc in test_docs : 
    print(f"found : {doc.page_content}")

Testing :space exploration company 

found : SpaceX develops Starship rockets for Mars missions.
found : Google is a large technology company with global operations.


### 2. BM25 RETRIEVER (Keyword Search/ Sparse Retrieval)

In [22]:
print("setting up bm25 retriever...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k=3

setting up bm25 retriever...


In [23]:
test_query = "Tesla"
print(f"Testing {test_query}")
test_docs  = bm25_retriever.invoke(test_query)
for doc in test_docs : 
    print(f"found : {doc.page_content}")
    

Testing Tesla
found : Tesla Tesla Tesla financial quarterly results improved significantly.
found : Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
found : Tesla Cybertruck production ramp begins in 2024.


### 3. Hybrid Retriever (Combination)

In [24]:
print("Setting up the hybrid retriever...")
hybrid_retriver = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights=[0.5,0.5]
)
print("setup completed \n")

Setting up the hybrid retriever...
setup completed 



In [25]:
test_query ="purchase cost 7.5 billion"
retrieved_chunks  = hybrid_retriver.invoke(test_query)
for i,doc in enumerate(retrieved_chunks,1) : 
    print(f"{i} : {doc.page_content}")
print()


1 : Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2 : Microsoft is a large technology company with global operations.
3 : Orange County reported new housing developments.
4 : Orange juice sales increased during winter months.

